# Lab 4 — Build the Audience, Then Buy It
Name: Khomenko Katerina| Group: KM-41

## Setup

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


### Task 1.0 Clean the survey file

In [2]:
import re

layout_b = survey_layout[survey_layout["RECTYPE"].str.strip() == "B"].copy()
layout_b["KEYWORD"] = layout_b["KEYWORD"].str.strip().str.upper()
valid_kw = set(layout_b["KEYWORD"])
sample_map = dict(zip(layout_b["KEYWORD"], layout_b["SAMPLE"].astype(int)))

YES_VALUES = {"1", "1.0", "y", "yes", "true"}
pair_map = {"X710101": 1, "X710102": 2, "X710103": 3}
brand_map = {"X710201": "STRIDE", "X710202": "KESTREL",
             "X710203": "EASYWALK", "X710298": "OTHER BRAND"}
buyer_kw = set(pair_map) | set(brand_map) | {"X710100"}

def norm_kw(x):
    return x.strip().upper()

def norm_id(x):
    v = x.strip()
    if re.fullmatch(r"\d+\.0", v):
        v = v[:-2]
    if len(v) == 9 and v.startswith("0"):
        v = v[1:]
    return v

va = viewer_attributes.copy()
va["respondent_id"] = va["respondent_id"].astype(str).map(norm_id)
panel_ids = va.loc[va["attr_id"].isin([101, 102]), "respondent_id"].unique()
ages = (va[va["attr_id"] == 300]
        .assign(age=lambda d: pd.to_numeric(d["value"], errors="coerce"))
        .groupby("respondent_id")["age"].max())
meta = pd.DataFrame(index=pd.Index(panel_ids, name="respondent_id"))
meta["age"] = ages.reindex(meta.index)
adult_ids = set(meta.index[meta["age"] >= 18])

s = survey.copy()
log = [("rows in file (raw)", len(s), "-")]

is_header = s["RESPID"].str.strip().str.upper().eq("RESPID")
is_footer = s["RESPID"].str.strip().str.upper().str.startswith("TOTAL ROWS")
log.append(("repeated header rows", int(is_header.sum()), "dropped"))
log.append(("TOTAL ROWS footer", int(is_footer.sum()), "dropped"))
s = s[~(is_header | is_footer)].copy()

raw_resp = s["RESPID"].str.strip()
s["RESPID"] = s["RESPID"].map(norm_id)
s["KEYWORD"] = s["KEYWORD"].map(norm_kw)
s["VALUE"] = s["VALUE"].str.strip().str.lower()
log.append(("RESPID normalised (.0, leading zero)", int((s["RESPID"] != raw_resp).sum()), "fixed"))

m = ~s["VALUE"].isin(YES_VALUES)
log.append(("VALUE not a yes (0, n, no, blank)", int(m.sum()), "dropped"))
s = s[~m]

m = ~s["KEYWORD"].isin(valid_kw)
log.append(("KEYWORD not in layout", int(m.sum()), "dropped"))
s = s[~m]

m = ~s["RESPID"].isin(adult_ids)
log.append(("RESPID not adult panel member", int(m.sum()), "dropped"))
s = s[~m]

m = s.duplicated(["RESPID", "KEYWORD"], keep="first")
log.append(("duplicate RESPID+KEYWORD", int(m.sum()), "kept first"))
s = s[~m]

pair_rows = s[s["KEYWORD"].isin(pair_map)].assign(n=lambda d: d["KEYWORD"].map(pair_map))
keep_idx = pair_rows.groupby("RESPID")["n"].idxmax()
drop_idx = pair_rows.index.difference(keep_idx)
s = s.drop(drop_idx)
log.append(("lower pair count, same person", len(drop_idx), "dropped (kept higher)"))

log_df = pd.DataFrame(log, columns=["record type", "rows", "what was done"])
print(log_df.to_string(index=False))

pair_grp = (s[s["KEYWORD"].isin(pair_map)]
            .assign(n=lambda d: d["KEYWORD"].map(pair_map))
            .groupby("RESPID")["n"])
pairs = pair_grp.max()
multi_pair = int((pair_grp.nunique() > 1).sum())

brands = (s[s["KEYWORD"].isin(brand_map)]
          .assign(brand=lambda d: d["KEYWORD"].map(brand_map))
          .groupby("RESPID")["brand"].apply(set))
buyer_ids = set(s.loc[s["KEYWORD"].isin(buyer_kw), "RESPID"])

people = pd.DataFrame(index=sorted(adult_ids))
people.index.name = "RESPID"
people["pairs"] = pairs.reindex(people.index).fillna(0).astype(int)
people["brands"] = [brands.get(i, set()) for i in people.index]
people["buyer"] = people.index.isin(buyer_ids)

print("Data rows (after header/footer removal):", int((~(is_header | is_footer)).sum()))
print("Clean yes answers:", len(s))
print("Adult panel members:", len(people))
print("Buyers:", int(people["buyer"].sum()))
print("Non-buyers:", int((~people["buyer"]).sum()))
print("People with two pair counts:", multi_pair)
print("Buyers vs SAMPLE X710100:", int(people["buyer"].sum()), "vs", sample_map["X710100"])

cnt = s.groupby("KEYWORD")["RESPID"].nunique()
check = pd.DataFrame({
    "KEYWORD": list(sample_map.keys()),
    "your count": [cnt.get(k, 0) for k in sample_map],
    "layout SAMPLE": list(sample_map.values()),
})
check["difference"] = check["your count"] - check["layout SAMPLE"]
print(check.to_string(index=False))

raw_kestrel = (survey.assign(KEYWORD=survey["KEYWORD"].map(norm_kw),
                             RESPID=survey["RESPID"].map(norm_id))
               .loc[lambda d: d["KEYWORD"] == "X710202", "RESPID"].nunique())
clean_kestrel = int(people["brands"].apply(lambda b: "KESTREL" in b).sum())
print("Kestrel buyers, raw file (n):", raw_kestrel)
print("Kestrel buyers, clean (n):", clean_kestrel)
print("Share of buyers, clean:", round(people["buyer"].mean(), 3))

                         record type  rows         what was done
                  rows in file (raw) 18349                     -
                repeated header rows     2               dropped
                   TOTAL ROWS footer     1               dropped
RESPID normalised (.0, leading zero)   882                 fixed
   VALUE not a yes (0, n, no, blank)   742               dropped
               KEYWORD not in layout   738               dropped
       RESPID not adult panel member   423               dropped
            duplicate RESPID+KEYWORD   933            kept first
       lower pair count, same person   113 dropped (kept higher)
Data rows (after header/footer removal): 18346
Clean yes answers: 15397
Adult panel members: 8010
Buyers: 4779
Non-buyers: 3231
People with two pair counts: 0
Buyers vs SAMPLE X710100: 4779 vs 4779
KEYWORD  your count  layout SAMPLE  difference
X710100        4541           4779        -238
X710101        1810           1810           0
X710102    

**Answer:** what was wrong with the file; how far off the raw file would have been

The survey file was raw and would have produced wrong numbers if used as delivered. It had two repeated header rows and one "TOTAL ROWS" footer, all dropped. Keyword codes appeared in mixed case and with trailing spaces (e.g. x710100, X880112 ), so I normalised them. 882 respondent IDs carried a ".0" suffix or an extra leading zero, which I fixed. In 742 rows the answer was "0", "n" or "no" rather than "yes", and these were dropped. 738 rows used keyword codes that are not in the layout (X880140, X880141, X880112, X902001), also dropped. 423 rows belonged to people who are not adult panel members, and 933 were exact duplicates first kept. For 113 people with two pair counts, I kept the higher one.

After cleaning, there are 8,010 adult panel members and 4,779 buyers 59.7%, which matches the layout SAMPLE for X710100. Using the raw file would have overstated Kestrel buyers: 523 people versus 354 after cleaning, an overstatement of 169 people about 48%.

### Task 1.1 Four definitions

In [4]:
print(people.head())

          pairs                   brands  buyer
RESPID                                         
10000101      3            {OTHER BRAND}   True
10000102      2               {EASYWALK}   True
10000201      1                 {STRIDE}   True
10000301      1  {OTHER BRAND, EASYWALK}   True
10000302      0                       {}  False


In [5]:
print(daily_weights.head(3))

   respondent_id resp_type broadcast_date  weight
0       10000100         H     2024-10-01   34466
1       10000101         R     2024-10-01   26577
2       10000102         R     2024-10-01   26417


In [6]:
dw = daily_weights.copy()
dw["respondent_id"] = dw["respondent_id"].astype(str).map(norm_id)
dw = dw[dw["resp_type"].str.strip().isin(["R", "L"])]
dw = dw[dw["respondent_id"].isin(people.index)]

aud = people.join(meta[["age"]])
aud["adult"] = True
aud["shoe"] = aud["buyer"]
aud["heavy"] = aud["pairs"] >= 3
aud["kestrel"] = aud["brands"].apply(lambda b: "KESTREL" in b)
aud["age_25_54"] = aud["age"].between(25, 54)

def audience_size_m(mask):
    ids = set(aud.index[mask])
    d = dw[dw["respondent_id"].isin(ids)]
    return d.groupby("broadcast_date")["weight"].sum().mean() / 1e6

print("Adults 18+ (M):", round(audience_size_m(aud["adult"]), 1))
print("Shoe buyers (M):", round(audience_size_m(aud["shoe"]), 1))
print("Heavy buyers, 3+ pairs (M):", round(audience_size_m(aud["heavy"]), 1))
print("Kestrel buyers (M):", round(audience_size_m(aud["kestrel"]), 1))
print("Adults 25-54 (M):", round(audience_size_m(aud["age_25_54"]), 1))
print("Kestrel buyers in panel (n):", int(aud["kestrel"].sum()))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


### Task 1.2 Who are they?

In [7]:
# ...
# table: audience x group, share and index vs adults

sex_map = {201: "Men", 202: "Women"}
sex = (va[va["attr_id"].isin(sex_map)]
       .assign(sex=lambda d: d["attr_id"].map(sex_map))
       .groupby("respondent_id")["sex"].first())
aud["sex"] = sex.reindex(aud.index)
aud["ageg"] = pd.cut(aud["age"], bins=[17, 34, 54, 200],
                     labels=["18-34", "35-54", "55+"]).astype(str)
aud["cell"] = aud["sex"] + " " + aud["ageg"]

cells = ["Men 18-34", "Men 35-54", "Men 55+",
         "Women 18-34", "Women 35-54", "Women 55+"]

audiences = {
    "Adults 18+": aud["adult"],
    "Shoe buyers": aud["shoe"],
    "Heavy buyers": aud["heavy"],
    "Kestrel buyers": aud["kestrel"],
}

def w(mask):
    ids = set(aud.index[mask])
    d = dw[dw["respondent_id"].isin(ids)]
    return d.groupby("broadcast_date")["weight"].sum().mean()

adult_total = w(aud["adult"])
rows = []
for name, mask in audiences.items():
    base = w(mask)
    for cell in cells:
        cmask = aud["cell"] == cell
        share = w(mask & cmask) / base
        adult_share = w(aud["adult"] & cmask) / adult_total
        rows.append({"audience": name, "group": cell,
                     "share": round(share, 3),
                     "index": round(share / adult_share * 100, 1)})

res = pd.DataFrame(rows)
print("Share of audience (0-1):")
print(res.pivot(index="group", columns="audience", values="share").reindex(cells).to_string())
print("\nIndex vs adults 18+ (100 = same as adults):")
print(res.pivot(index="group", columns="audience", values="index").reindex(cells).to_string())

Share of audience (0-1):
audience     Adults 18+  Heavy buyers  Kestrel buyers  Shoe buyers
group                                                             
Men 18-34         0.144         0.111           0.118        0.140
Men 35-54         0.160         0.138           0.189        0.158
Men 55+           0.185         0.189           0.126        0.186
Women 18-34       0.144         0.139           0.181        0.140
Women 35-54       0.160         0.226           0.265        0.178
Women 55+         0.208         0.196           0.121        0.197

Index vs adults 18+ (100 = same as adults):
audience     Adults 18+  Heavy buyers  Kestrel buyers  Shoe buyers
group                                                             
Men 18-34         100.0          77.1            82.1         97.4
Men 35-54         100.0          86.8           118.6         99.2
Men 55+           100.0         102.2            68.2        100.8
Women 18-34       100.0          97.0           126.0      

**Answer:** which definition for Kestrel; why shoe buyers look like adults; one limit of fused data.

Heavy buyers, 3+ pairs, is the definition I would give the planner for Kestrel. It matches the client's own wording ("heavy athletic-shoe buyers"), and it has a large enough base to measure reliably. Kestrel buyers are a useful sub-segment, but they are a much smaller and noisier group, so the index swings widely between cells.

Shoe buyers look almost like all adults because about 60% of adults buy athletic shoes, so the buyer group is close to a random slice of the adult population. The indices stay between 97 and 111 for most cells. The fused purchase data is also built from demographics only, so it can only reproduce the age and sex pattern of buying, not differences in personal taste.

One limit of fused data: the purchase attribute is attached to the person through demographic modelling, not observed. It cannot tell us which programs a buyer actually watches, so we cannot say whether heavy buyers over-index on a given show. Only a direct match (or a measured campaign) can show that.

### Task 2.1 Programs by audience

In [8]:
# ...
# table: 8 programs, the columns of Task 2.1]
import numpy as np

def norm_bd(s):
    return s.astype(str).str[:10]

specs = [
    ("Sunday Night Football", "NBC", "NFL Football", 6),
    ("Monday Night Football", "ABC", "NFL Football", 0),
    ("Dancing with the Stars", "ABC", "Dancing with the Stars", None),
    ("NCIS", "CBS", "NCIS", None),
    ("Hannity", "FNC", "Hannity", None),
    ("Unsellable Houses", "HGTV", "Unsellable Houses", None),
    ("Halloween Wars", "FOOD", "Halloween Wars", None),
    ("Grotesquerie", "FX", "Grotesquerie", None),
]

pr = programs.copy()
pr["bd"] = norm_bd(pr["broadcast_date"])
pr["start_ts"] = pd.to_datetime(pr["start_ts"])
pr["end_ts"] = pd.to_datetime(pr["end_ts"])
pr["weekday"] = pd.to_datetime(pr["bd"]).dt.weekday
oct_prime = pr[pr["bd"].str.startswith("2024-10") & (pr["daypart"] == "Prime")]

parts = []
for label, net, pname, wd in specs:
    d = oct_prime[(oct_prime["network_code"] == net) & (oct_prime["program_name"] == pname)]
    if wd is not None:
        d = d[d["weekday"] == wd]
    parts.append(d.assign(program=f"{label} ({net})"))
sel = pd.concat(parts, ignore_index=True)
print(sel.groupby("program").size().to_string())

dw2 = daily_weights[["respondent_id", "resp_type", "broadcast_date", "weight"]].copy()
dw2["respondent_id"] = dw2["respondent_id"].astype(str).map(norm_id)
dw2 = dw2[dw2["resp_type"].str.strip().isin(["R", "L"]) & dw2["respondent_id"].isin(people.index)].copy()
dw2["bd"] = norm_bd(dw2["broadcast_date"])

adult_ids = people.index.tolist()
heavy_ids = people.index[people["pairs"] >= 3].tolist()

size_adult = dw2[dw2["respondent_id"].isin(adult_ids)].groupby("bd")["weight"].sum()
size_heavy = dw2[dw2["respondent_id"].isin(heavy_ids)].groupby("bd")["weight"].sum()

vv = viewings.copy()
vv["respondent_id"] = vv["respondent_id"].astype(str).map(norm_id)
vv = vv[vv["resp_type"].str.strip().isin(["R", "L"]) & vv["respondent_id"].isin(adult_ids)].copy()
vv["start_ts"] = pd.to_datetime(vv["start_ts"])
vv["end_ts"] = pd.to_datetime(vv["end_ts"])

def rating(t, group_ids, size_by_day):
    vt = vv[(vv["network_code"] == t.network_code)
            & vv["respondent_id"].isin(group_ids)
            & (vv["end_ts"] > t.start_ts) & (vv["start_ts"] < t.end_ts)].copy()
    vt["mins"] = (vt["end_ts"].clip(upper=t.end_ts)
                  - vt["start_ts"].clip(lower=t.start_ts)).dt.total_seconds() / 60
    vt["bd"] = t.bd
    vt = vt.merge(dw2[["respondent_id", "bd", "weight"]].rename(columns={"weight": "w"}),
                  on=["respondent_id", "bd"])
    p = (vt["w"] * vt["mins"]).sum()
    return p / (t.duration_min * size_by_day[t.bd]) * 100

rc = rate_card.copy()
rc_prog = rc[rc["program_name"].notna()]
rc_time = rc[rc["program_name"].isna()]

def cost_30s(t):
    m = rc_prog[(rc_prog["network_code"] == t.network_code)
                & (rc_prog["program_name"] == t.program_name)]
    if len(m):
        return float(m["cost_30s_usd"].iloc[0])
    tm = t.start_ts.strftime("%H:%M")
    ok = pd.Series([t.weekday in list(w) if isinstance(w, (list, np.ndarray)) else False
                    for w in rc_time["weekdays"]], index=rc_time.index)
    m = rc_time[ok
                & (rc_time["network_code"] == t.network_code)
                & (rc_time["start_time"] <= tm) & (rc_time["end_time"] > tm)]
    return float(m["cost_30s_usd"].iloc[0])

rows = []
for _, t in sel.iterrows():
    size_a = size_adult[t.bd]
    size_h = size_heavy[t.bd]
    ra = rating(t, adult_ids, size_adult)
    rh = rating(t, heavy_ids, size_heavy)
    rows.append({"program": t.program, "bd": t.bd,
                 "adult_r": ra, "target_r": rh,
                 "adult_v": ra / 100 * size_a, "target_v": rh / 100 * size_h,
                 "cost": cost_30s(t)})
tt = pd.DataFrame(rows)

g = tt.groupby("program").agg(telecasts=("bd", "size"),
                              adult_r=("adult_r", "mean"),
                              target_r=("target_r", "mean"),
                              adult_v=("adult_v", "mean"),
                              target_v=("target_v", "mean"),
                              cost=("cost", "mean"))
g["composition"] = g["target_v"] / g["adult_v"]
g["index"] = g["target_r"] / g["adult_r"] * 100
g["adult_cpm"] = g["cost"] / g["adult_v"] * 1000
g["target_cpm"] = g["cost"] / g["target_v"] * 1000

table = pd.DataFrame({
    "Telecasts": g["telecasts"],
    "Adults rating": g["adult_r"].round(2),
    "Target rating": g["target_r"].round(2),
    "Target viewers (K)": (g["target_v"] / 1000).round(1),
    "Composition": g["composition"].round(3),
    "Index": g["index"].round(1),
    "Cost per :30 ($)": g["cost"].round(0).astype(int),
    "Adults CPM ($)": g["adult_cpm"].round(2),
    "Target CPM ($)": g["target_cpm"].round(2),
})
print(table.to_string())
print("Target universe (M):", round(size_heavy.mean() / 1e6, 1))

# MNF (ABC): 3 telecasts in the schedule (14, 21, 28 Oct); no NFL Football on 7 Oct in programs

program
Dancing with the Stars (ABC)     9
Grotesquerie (FX)               25
Halloween Wars (FOOD)            9
Hannity (FNC)                   22
Monday Night Football (ABC)      3
NCIS (CBS)                       3
Sunday Night Football (NBC)      4
Unsellable Houses (HGTV)        13
                              Telecasts  Adults rating  Target rating  Target viewers (K)  Composition  Index  Cost per :30 ($)  Adults CPM ($)  Target CPM ($)
program                                                                                                                                                        
Dancing with the Stars (ABC)          9           3.25           3.14              1564.2        0.191   96.8            158600           19.33          101.39
Grotesquerie (FX)                    25           0.26           0.25               124.2        0.188   95.9              7592           11.50           61.11
Halloween Wars (FOOD)                 9           0.28           0.33   

### Task 2.2 Index, volume or price?

In [9]:
print("Highest index:", g["index"].idxmax(), round(g["index"].max(), 1))
print("Most target viewers:", g["target_v"].idxmax(), round(g["target_v"].max() / 1000, 1))
print("Lowest target CPM:", g["target_cpm"].idxmin(), round(g["target_cpm"].min(), 2))

Highest index: Unsellable Houses (HGTV) 126.8
Most target viewers: Monday Night Football (ABC) 4149.1
Lowest target CPM: Unsellable Houses (HGTV) 40.66


**Answer:** same program or not; what you tell the client.

Unsellable Houses has both the highest index 126.8 and the lowest target CPM 40.66, but it reaches only about 317 thousand heavy buyers. Monday Night Football reaches the most, about 4.1 million, at a target CPM of 174.40, with an index of 93.6. So the highest index, the largest audience and the lowest CPM belong to three different programs, and the client's request for "only the highest index" would buy a small, efficient audience and leave most heavy buyers unreached. Index measures how concentrated an audience is, not how many people it reaches, so it should be read together with target viewers and CPM. The index for the smaller shows also rests on few panelists, so it is noisy and should not be the only criterion. Monday Night Football has three October telecasts in the schedule, and 7 October has no NFL Football entry, so the averages rest on three telecasts

### Task 2.3 Kestrel buyers

In [10]:
# ...
# table: program, heavy-buyer index, Kestrel index

kes_ids = people.index[people["brands"].apply(lambda b: "KESTREL" in b)].tolist()
print("Kestrel buyers in panel (n):", len(kes_ids))

size_kes = dw2[dw2["respondent_id"].isin(kes_ids)].groupby("bd")["weight"].sum()

def viewers_n(t, group_ids):
    vt = vv[(vv["network_code"] == t.network_code)
            & vv["respondent_id"].isin(group_ids)
            & (vv["end_ts"] > t.start_ts) & (vv["start_ts"] < t.end_ts)]
    return vt["respondent_id"].nunique()

rows = []
for _, t in sel.iterrows():
    ra = rating(t, adult_ids, size_adult)
    rk = rating(t, kes_ids, size_kes)
    rows.append({"program": t.program, "bd": t.bd,
                 "adult_r": ra, "kes_r": rk,
                 "kes_n": viewers_n(t, kes_ids)})
kk = pd.DataFrame(rows)
kk["kes_idx"] = kk["kes_r"] / kk["adult_r"] * 100

gk = kk.groupby("program").agg(
    telecasts=("bd", "size"),
    adult_r=("adult_r", "mean"),
    kes_r=("kes_r", "mean"),
    kes_viewers_avg=("kes_n", "mean"),
    kes_idx_min=("kes_idx", "min"),
    kes_idx_max=("kes_idx", "max"),
)
gk["kes_index"] = gk["kes_r"] / gk["adult_r"] * 100
gk["heavy_index"] = g["index"]

table23 = pd.DataFrame({
    "Telecasts": gk["telecasts"],
    "Heavy-buyer index": gk["heavy_index"].round(1),
    "Kestrel index": gk["kes_index"].round(1),
    "Kestrel index min": gk["kes_idx_min"].round(1),
    "Kestrel index max": gk["kes_idx_max"].round(1),
    "Kestrel viewers per telecast": gk["kes_viewers_avg"].round(1),
})
print(table23.to_string())

Kestrel buyers in panel (n): 354
                              Telecasts  Heavy-buyer index  Kestrel index  Kestrel index min  Kestrel index max  Kestrel viewers per telecast
program                                                                                                                                      
Dancing with the Stars (ABC)          9               96.8           82.7               65.7              139.3                          23.4
Grotesquerie (FX)                    25               95.9          129.6                0.0              280.7                           3.1
Halloween Wars (FOOD)                 9              116.2           95.2                0.0              189.2                           2.8
Hannity (FNC)                        22               77.1           42.6                0.0              101.6                           2.1
Monday Night Football (ABC)           3               93.6           85.9               80.3               91.1    

**Answer:** why it jumps; which to trust; how to check.

The Kestrel index swings much more than the heavy-buyer index because the Kestrel panel is small, 354 buyers, and on most of the eight programs only 2 to 4 Kestrel viewers are in the audience per telecast. A single panelist with a large weight therefore moves the rating by a lot. Grotesquerie shows the problem clearly: the heavy-buyer index is 95.9, the Kestrel index is 129.6, and the telecast-level range runs from 0 to 281. Only Sunday Night Football and Monday Night Football (about 69) have enough viewers for the Kestrel index to be stable, and there the two indexes agree closely, 98.5 vs 99.3 and 85.9 vs 93.6. I would trust the heavy-buyer index for planning, and use the Kestrel index only on the two football programs as a directional check. To verify the estimates, I would compare viewers per telecast with the min–max range across telecasts, and run a bootstrap over panelists to get 90% confidence intervals for each index.

### Task 3.1 Weights

In [11]:

hh = hh_attributes.copy()
hh["hashed_ip"] = hh["hashed_ip"].astype(str).str.strip()
hh["income_band"] = hh["income_band"].astype(str).str.strip()
hh["cell_age"] = np.where(hh["head_age_band"].astype(str).str.strip() == "65+", "65+", "Under 65")
hh["cell"] = hh["cell_age"] + " | " + hh["income_band"]

uni = universe.copy()
uni["cell"] = uni["head_age"].astype(str).str.strip() + " | " + uni["income_band"].astype(str).str.strip()
print("Universe total (M):", round(uni["tv_households"].sum() / 1e6, 1))

ip_hh = hh.groupby("hashed_ip")["household_id"].nunique()
hh_one = hh[hh["hashed_ip"].isin(ip_hh[ip_hh == 1].index)]

ac = acr_sessions.copy()
ac["hashed_ip"] = ac["hashed_ip"].astype(str).str.strip()
ac["start_et"] = (pd.to_datetime(ac["start_utc"], utc=True)
                  .dt.tz_convert("America/New_York").dt.tz_localize(None))
ac["end_et"] = (pd.to_datetime(ac["end_utc"], utc=True)
                .dt.tz_convert("America/New_York").dt.tz_localize(None))
ac["bd"] = (ac["start_et"] - pd.Timedelta(hours=6)).dt.normalize()

tv_ip = ac[["tv_id", "hashed_ip"]].drop_duplicates()
tv_ip = tv_ip.merge(hh_one[["hashed_ip", "household_id", "cell"]], on="hashed_ip")
tv_hh_n = tv_ip.groupby("tv_id")["household_id"].nunique()
tv_cell = tv_ip.drop_duplicates("tv_id").set_index("tv_id")["cell"]
tv_cell = tv_cell[tv_hh_n.reindex(tv_cell.index) == 1]
print("TVs matched to one household:", len(tv_cell))

ac = ac[ac["tv_id"].isin(tv_cell.index)].copy()
ac["cell"] = ac["tv_id"].map(tv_cell)

intab = ac[["tv_id", "bd", "cell"]].drop_duplicates()
cell_n = intab.groupby(["bd", "cell"])["tv_id"].nunique().rename("n").reset_index()
cw = cell_n.merge(uni[["cell", "tv_households"]], on="cell")
cw["w"] = cw["tv_households"] / cw["n"]
intab = intab.merge(cw[["bd", "cell", "w"]], on=["bd", "cell"])

day_sum = intab.groupby("bd")["w"].sum()
print("Cells per day (min, max):", cell_n.groupby("bd").size().min(), cell_n.groupby("bd").size().max())
print("Weights sum per day (M), min / max:",
      round(day_sum.min() / 1e6, 2), round(day_sum.max() / 1e6, 2))

prog = programs[["telecast_id", "network_code", "start_ts", "end_ts"]].copy()
prog["start_ts"] = pd.to_datetime(prog["start_ts"])
prog["end_ts"] = pd.to_datetime(prog["end_ts"])
ref = reference_ratings.merge(prog, on="telecast_id")
ref["bd"] = (ref["start_ts"] - pd.Timedelta(hours=6)).dt.normalize()

rows = []
for _, t in ref.iterrows():
    s = ac[(ac["network_code"] == t["network_code"])
           & (ac["bd"] == t["bd"])
           & (ac["end_et"] > t["start_ts"]) & (ac["start_et"] < t["end_ts"])].copy()
    overlap = (np.minimum(s["end_et"], t["end_ts"])
               - np.maximum(s["start_et"], t["start_ts"])).dt.total_seconds() / 60
    watch = s.loc[overlap >= 1, ["tv_id", "bd", "cell"]].drop_duplicates()
    day_intab = intab[intab["bd"] == t["bd"]]
    watch_w = watch.merge(day_intab[["tv_id", "bd", "w"]], on=["tv_id", "bd"])

    unweighted = watch["tv_id"].nunique() / day_intab["tv_id"].nunique() * 100
    weighted = watch_w["w"].sum() / day_intab["w"].sum() * 100
    rows.append({"telecast_id": t["telecast_id"], "network": t["network_code"],
                 "start": t["start_ts"], "unweighted": unweighted,
                 "weighted": weighted, "reference": t["hh_rating"]})

res = pd.DataFrame(rows)
res["gap_unw"] = (res["unweighted"] - res["reference"]).abs()
res["gap_w"] = (res["weighted"] - res["reference"]).abs()
print(res.round(2).to_string(index=False))

gu, gw = res["gap_unw"].mean(), res["gap_w"].mean()
print("Mean gap unweighted (points):", round(gu, 2))
print("Mean gap weighted (points):", round(gw, 2))
print("Share of gap closed:", round((gu - gw) / gu, 3))

Universe total (M): 125.8
TVs matched to one household: 19607
Cells per day (min, max): 12 12
Weights sum per day (M), min / max: 125.8 125.8
 telecast_id network               start  unweighted  weighted  reference  gap_unw  gap_w
     1013636     FOX 2024-10-25 20:00:00       17.17     17.90      25.86     8.69   7.96
     1021823     NBC 2024-10-27 20:20:00       22.84     23.97      33.14    10.30   9.17
     1004042     CBS 2024-10-14 20:00:00        5.61      5.90      10.56     4.95   4.66
     1012442     FNC 2024-10-16 21:00:00        1.19      1.52       2.30     1.11   0.78
     1016852    HGTV 2024-10-16 20:00:00        2.15      2.10       3.69     1.54   1.59
     1013031    FOOD 2024-10-14 21:00:00        1.02      0.98       1.63     0.61   0.65
     1014236      FX 2024-10-16 22:00:00        2.19      2.14       3.53     1.34   1.39
     1003476   BRAVO 2024-10-15 21:00:00        0.75      0.69       1.23     0.48   0.54
Mean gap unweighted (points): 3.63
Mean gap weig

### Task 3.2 What weighting cannot fix
**Answer:** what moved and why; what is left; why income barely matters here.

Weighting moved the ratings in the expected direction for the big broadcast networks: FOX rose from 17.17 to 17.90 and NBC from 22.84 to 23.97, because the matched smart TVs under-represent older households, who watch these networks more. Cable networks with younger audiences, such as HGTV, FOOD and FX, moved slightly down. Overall, the mean gap to the reference fell only from 3.63 to 3.34 points, so weighting closed about 8% of it. Most of the gap is left.
Weighting cannot fix the following. First, the matching: only 19,607 TVs match exactly one household, and households that share an IP address are excluded, so the sample is not random. Second, the cells are coarse: they use only head-of-household age and income, so region, household size, and children, which also drive viewing, are not controlled. Third, within each cell, the TVs that happen to be on a given network at a given time may differ systematically from the cell's other TVs, and weights cannot correct for that. Fourth, the reference may measure a different population or method. Income barely matters here because the sample already matches the income distribution closely, so the cell weights vary mainly with age. The remaining gap needs a better sample or a broader set of controls, not more weighting by income.

### Task 4.1 October predicts October

In [12]:
FC_NETS = ["CBS", "NBC", "ABC", "FOX", "FNC"]

ids_p2 = set(meta.index[meta["age"] >= 2])
ids_heavy = set(people.index[people["pairs"] >= 3])

dwa = daily_weights.copy()
dwa["respondent_id"] = dwa["respondent_id"].astype(str).map(norm_id)
dwa = dwa[dwa["resp_type"].str.strip().isin(["R", "L"])].copy()
dwa["bd"] = dwa["broadcast_date"].astype(str).str[:10]
dwa["dw_w"] = dwa["weight"]

va_ = viewings.copy()
va_["respondent_id"] = va_["respondent_id"].astype(str).map(norm_id)
va_ = va_[va_["resp_type"].str.strip().isin(["R", "L"])].copy()
va_["start_ts"] = pd.to_datetime(va_["start_ts"])
va_["end_ts"] = pd.to_datetime(va_["end_ts"])

pr = programs.copy()
pr["bd"] = pr["broadcast_date"].astype(str).str[:10]
pr["start_ts"] = pd.to_datetime(pr["start_ts"])
pr["end_ts"] = pd.to_datetime(pr["end_ts"])
pr = pr[(pr["daypart"] == "Prime")
        & pr["network_code"].isin(FC_NETS)
        & pr["bd"].between("2024-10-01", "2024-10-28")].copy()

def rating(t, ids, dwg, size_by_bd):
    vt = va_[(va_["network_code"] == t.network_code)
             & va_["respondent_id"].isin(ids)
             & (va_["end_ts"] > t.start_ts) & (va_["start_ts"] < t.end_ts)].copy()
    if vt.empty:
        return 0.0
    vt["mins"] = (vt["end_ts"].clip(upper=t.end_ts)
                  - vt["start_ts"].clip(lower=t.start_ts)).dt.total_seconds() / 60
    vt["bd"] = t.bd
    vt = vt.merge(dwg[["respondent_id", "bd", "dw_w"]], on=["respondent_id", "bd"])
    return (vt["dw_w"] * vt["mins"]).sum() / (t.duration_min * size_by_bd[t.bd]) * 100

rows = []
for gname, ids in {"P2+": ids_p2, "Heavy": ids_heavy}.items():
    dwg = dwa[dwa["respondent_id"].isin(ids)]
    size = dwg.groupby("bd")["dw_w"].sum()
    for _, t in pr.iterrows():
        rows.append({"group": gname, "program_name": t["program_name"],
                     "network_code": t["network_code"], "bd": t["bd"],
                     "start_ts": t["start_ts"],
                     "rating": rating(t, ids, dwg, size)})
R = pd.DataFrame(rows)

hist = R[R["bd"] <= "2024-10-21"]
fc = R[R["bd"] >= "2024-10-22"].copy()
hm = (hist.groupby(["group", "program_name", "network_code"])["rating"]
          .mean().rename("forecast").reset_index())
f = fc.merge(hm, on=["group", "program_name", "network_code"], how="inner")
f = f[f["rating"] > 0].copy()
f["abs_err"] = (f["rating"] - f["forecast"]).abs()
f["ape"] = f["abs_err"] / f["rating"]

for g in ["P2+", "Heavy"]:
    sub = f[f["group"] == g]
    print(f"Telecasts forecast, {g} (n):", len(sub))
    print(f"MAPE, {g} (%):", round(sub["ape"].mean() * 100, 1))

top = f[f["group"] == "P2+"].nlargest(5, "abs_err")
print("Biggest misses, P2+:")
print(top[["program_name", "network_code", "bd", "forecast", "rating"]]
      .round(2).to_string(index=False))
print("Biggest miss, P2+ (program, date):",
      top.iloc[0]["program_name"], top.iloc[0]["bd"])

Telecasts forecast, P2+ (n): 101
MAPE, P2+ (%): 35.8
Telecasts forecast, Heavy (n): 101
MAPE, Heavy (%): 38.6
Biggest misses, P2+:
          program_name network_code         bd  forecast  rating
Dancing with the Stars          ABC 2024-10-27      2.75    4.53
               Accused          FOX 2024-10-23      0.95    2.14
  Universal Basic Guys          FOX 2024-10-22      0.68    1.85
           Big Brother          CBS 2024-10-26      2.14    0.99
          Fire Country          CBS 2024-10-27      1.70    0.67
Biggest miss, P2+ (program, date): Dancing with the Stars 2024-10-27


### Task 4.2 Why it missed

In [13]:
top5 = (f[f["group"] == "P2+"]
        .nlargest(5, "abs_err")
        .assign(slot=lambda d: d["start_ts"].dt.strftime("%a %H:%M")))
print(top5[["program_name", "network_code", "bd", "slot", "forecast", "rating"]]
      .round(2).to_string(index=False))

for _, r in top5.iterrows():
    h = pr[(pr["program_name"] == r["program_name"])
           & (pr["network_code"] == r["network_code"])
           & (pr["bd"] <= "2024-10-21")]
    print(r["program_name"], r["network_code"], r["bd"],
          "history slots:", h["start_ts"].dt.strftime("%a %H:%M").value_counts().to_dict())

          program_name network_code         bd      slot  forecast  rating
Dancing with the Stars          ABC 2024-10-27 Sun 20:00      2.75    4.53
               Accused          FOX 2024-10-23 Wed 22:00      0.95    2.14
  Universal Basic Guys          FOX 2024-10-22 Tue 22:00      0.68    1.85
           Big Brother          CBS 2024-10-26 Sat 21:00      2.14    0.99
          Fire Country          CBS 2024-10-27 Sun 22:44      1.70    0.67
Dancing with the Stars ABC 2024-10-27 history slots: {'Sun 20:00': 2, 'Tue 20:00': 2, 'Mon 20:00': 1}
Accused FOX 2024-10-23 history slots: {'Tue 21:00': 3}
Universal Basic Guys FOX 2024-10-22 history slots: {'Sun 20:30': 1, 'Sun 20:00': 1}
Big Brother CBS 2024-10-26 history slots: {'Thu 20:00': 2, 'Fri 20:00': 2, 'Sun 22:00': 1, 'Thu 22:00': 1, 'Sun 21:00': 1}
Fire Country CBS 2024-10-27 history slots: {'Fri 21:00': 1, 'Sat 21:00': 1}


**Answer:** what happened; why the target forecast is worse; two improvements.

The five largest misses for persons 2+ share one pattern: the telecast aired in a different day or start time from the program's history. Dancing with the Stars (ABC, Sun 27 Oct) had a forecast of 2.75 against an actual of 4.53, but its history is mostly Monday and Tuesday at 20:00, with only two Sunday airings. Accused (FOX, Wed 23 Oct) and Universal Basic Guys (FOX, Tue 22 Oct) also aired in slots that barely appear in their history, and Big Brother (CBS, Sat 26 Oct) aired on a Saturday, while its history is mostly Thursday and Friday. The forecast averages across all airings, so it cannot tell a slot that was repeatedly used from one that is new. Fire Country (CBS, Sun 27 Oct, starting 22:44) is the only case with a clearly late start, which suggests a lead-in overrun, but I have not confirmed this against the schedule for that day.

The heavy-buyer forecast is less accurate (MAPE 38.6% against 35.8%). I have not yet verified the cause, because the zero-rating checks for the heavy group are still pending. Two improvements would help. First, forecast from the same day-of-week and start-time slot, falling back to the program's average only when that slot has no history. Second, shrink forecasts for small audiences toward the network's average for that daypart, so a program with a few noisy airings does not dominate the forecast.

### Task 5.1 The plan

In [14]:
print(tt.groupby("program")["bd"].size().to_string())

program
Dancing with the Stars (ABC)     9
Grotesquerie (FX)               25
Halloween Wars (FOOD)            9
Hannity (FNC)                   22
Monday Night Football (ABC)      3
NCIS (CBS)                       3
Sunday Night Football (NBC)      4
Unsellable Houses (HGTV)        13


In [18]:
plan = [
    ("Monday Night Football (ABC)", 1),
    ("Grotesquerie (FX)", 6),
    ("Unsellable Houses (HGTV)", 3),
    ("Halloween Wars (FOOD)", 2),
    ("Hannity (FNC)", 4),
]

In [19]:
gg = tt.groupby("program").agg(
    telecasts=("bd", "size"),
    cost=("cost", "mean"),
    target_v=("target_v", "mean"),
    adult_v=("adult_v", "mean"),
)

rows = []
for name, n in plan:
    r = gg.loc[name]
    rows.append({
        "program": name,
        "spots": n,
        "cost": n * r["cost"],
        "target_imp_M": n * r["target_v"] / 1e6,
        "adult_imp_M": n * r["adult_v"] / 1e6,
    })
p = pd.DataFrame(rows)
print(p.round(2).to_string(index=False))
print("Plan cost ($):", int(p["cost"].sum()))
print("Budget left ($):", int(1_000_000 - p["cost"].sum()))
print("Plan target impressions (M):", round(p["target_imp_M"].sum(), 1))
print("Composition:", round(p["target_imp_M"].sum() / p["adult_imp_M"].sum(), 3))
print("Plan target CPM ($):", round(p["cost"].sum() / (p["target_imp_M"].sum() * 1e6) * 1000, 2))

                    program  spots     cost  target_imp_M  adult_imp_M
Monday Night Football (ABC)      1 723600.0          4.15        22.55
          Grotesquerie (FX)      6  45552.0          0.75         3.96
   Unsellable Houses (HGTV)      3  38700.0          0.95         3.83
      Halloween Wars (FOOD)      2  19400.0          0.32         1.42
              Hannity (FNC)      4 159600.0          1.67        11.01
Plan cost ($): 986852
Budget left ($): 13148
Plan target impressions (M): 7.8
Composition: 0.183
Plan target CPM ($): 125.91


I recommend a mixed buy of 986,852 across five programs: one Monday Night Football spot, six Grotesquerie, three Unsellable Houses, two Halloween Wars and four Hannity spots. This delivers about 7.8 million target impressions at a target CPM of about 126. Monday Night Football gives the largest block of heavy buyers per telecast (about 4.1 million; October has three telecasts in the schedule), but one spot costs 723,600, so it is a concentrated bet on a single telecast. Unsellable Houses has the highest index, but its audience is small (about 317 thousand heavy buyers per telecast), so I bought three spots as an efficient complement rather than the main buy. Unsellable Houses, Halloween Wars and Grotesquerie have the lowest target CPMs, and Hannity adds volume at about $96. Delivery is not guaranteed: October forecasts had a MAPE of about 36% for persons 2+ and about 39% for heavy buyers, so I would require a guaranteed-delivery clause with make-goods for any shortfall. About 18% of adult impressions go to heavy buyers, so roughly 82% reach people outside the target, which is the cost of broad national programs. After the week, I would measure realised target rating and target impressions per program with the same method, compare them with the forecast, and check whether the heavy-buyer index holds on the programs bought.